# 01. Nuanic Ring Quickstart: Real-time Streaming & Telemetry

This tutorial demonstrates how to:
1. Discover visible Nuanic / Moodmetric rings via Bluetooth Low Energy (BLE).
2. Connect and configure operational modes (e.g. `MODE_LIVE` vs `MODE_RAW_EDA`).
3. Subscribe to physiological signals (Electrodermal Activity & DNE Stress Index) and IMU motion streams.
4. Capture data and inspect the generated session directory and provenance manifest (`session_manifest.json`).

In [ ]:
import asyncio
from pathlib import Path
from nuanic_ring.core import RingScanner, NuanicConnector
from nuanic_ring import NuanicMonitor, MODE_LIVE

## 1. Scan for Rings
We use `RingScanner` to search for nearby rings.

In [ ]:
scanner = RingScanner(timeout=5.0)
rings = await scanner.list_available_rings(stop_if_found=False)
print(f"Found {len(rings)} ring(s):")
for r in rings:
    print(f" - {r['name']} ({r['address']}) | RSSI: {r.get('rssi', 'N/A')}")

## 2. Initialize and Start Multi-Ring Monitor
`NuanicMonitor` manages BLE subscriptions, hardware clock timestamp smoothing, sample-rate diagnostics, and asynchronous CSV streaming.

In [ ]:
# Configure monitor with split raw/computed CSV logging and digital filtering
monitor = NuanicMonitor(
    log_dir="data/quickstart_logs",
    csv_layout="split",
    apply_filter=True,
    initial_mode=MODE_LIVE,
    participant_id="P01"
)

# Start monitoring all discovered rings (or provide specific MAC addresses)
ok = await monitor.start_multi(monitor_all=True)
print(f"Monitoring started: {ok}")

## 3. Live Telemetry Inspection
Let's stream for 10 seconds and inspect the live telemetry state.

In [ ]:
for _ in range(5):
    await asyncio.sleep(2.0)
    for row in monitor.dashboard_rows():
        print(f"MAC: {row['device_mac']} | Bat: {row['battery']} | EDA: {row['raw_eda']} | "
              f"Filt: {row['filtered_us']} µS | DNE: {row['dne_score']} | IMU: {row['imu_xyz']}")

## 4. Insert Event Markers and Stop Session
Event markers allow precise synchronization with external stimuli or experimental trials.

In [ ]:
monitor.add_marker("BASELINE_START", source="notebook")
await asyncio.sleep(2.0)
monitor.add_marker("BASELINE_END", source="notebook")

# Stop monitoring and flush queues to disk
await monitor.stop_multi()
print("Session stopped. Manifest and CSV logs written.")